# BOOTSTRAPING


## 1. Tomar dataset Moto Trend Road

In [4]:
pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.7 MB/s eta 0:00:00


In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from skopt import BayesSearchCV
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_excel("/content/Motor Trend Car Road Tests.xlsx")

In [ ]:
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [ ]:
df.columns

Index(['model', 'mpg', 'cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am',
       'gear', 'carb'],
      dtype='object')

## 2. Realizar regresión lineal sobre mpg = B0 + B1(hp) + B2(qsesc)

In [6]:
X = df[['hp', 'qsec']]
y = df['mpg']

In [ ]:
X_sm = sm.add_constant(X)
model = sm.OLS(y, X_sm)
model_fit = model.fit()


## 2a. Calcular intervalos de confianza de los betas

In [ ]:
print(model_fit.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 27 Apr 2026   Prob (F-statistic):           4.18e-07
Time:                        23:08:20   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

In [ ]:
print('Intervalos de confianza de los coeficientes (betas):')
print(model_fit.conf_int())

Intervalos de confianza de los coeficientes (betas):
               0          1
const  25.614894  71.032516
hp     -0.113089  -0.056097
qsec   -1.979929   0.206770


## 3. Crear 1000 muestras de boostrap con m observaciones con reemplazo


In [ ]:
n_bootstrap = 1000
m = len(df)

betas_bootstrap = []

for _ in range(n_bootstrap):
    muestra = df.sample(n=m, replace=True)

    X_muestra = muestra[['hp', 'qsec']]
    y_muestra = muestra['mpg']

    ##3a. Realizar la regresion mpg = B0 + B1(hp) + B2(qsesc)
    X_muestra_sm = sm.add_constant(X_muestra)
    modelo_b = sm.OLS(y_muestra, X_muestra_sm)
    modelo_b_fit = modelo_b.fit()

    betas_bootstrap.append(modelo_b_fit.params)

betas_df = pd.DataFrame(betas_bootstrap, columns=['B0', 'B1', 'B2'])

## 3b. Calcular Bx y O_Bx, abrir intervalos de confianza




In [ ]:
medias = betas_df.mean()
desv_std = betas_df.std()
ic_bootstrap = betas_df.quantile([0.025, 0.975])

## 4. Comparar resultados entre 2 y 3




In [ ]:
# Paso 4
ic_ols = model_fit.conf_int()

comparacion = pd.DataFrame({
    'OLS_beta':          model_fit.params.values,
    'OLS_IC_low':        ic_ols[0].values,
    'OLS_IC_high':       ic_ols[1].values,
    'Bootstrap_beta':    medias.values,
    'Bootstrap_std':     desv_std.values,
    'Bootstrap_IC_low':  ic_bootstrap.loc[0.025].values,
    'Bootstrap_IC_high': ic_bootstrap.loc[0.975].values,
}, index=['B0', 'B1', 'B2'])

print(comparacion.round(4))

    OLS_beta  OLS_IC_low  OLS_IC_high  Bootstrap_beta  Bootstrap_std  \
B0   48.3237     25.6149      71.0325             NaN            NaN   
B1   -0.0846     -0.1131      -0.0561             NaN            NaN   
B2   -0.8866     -1.9799       0.2068             NaN            NaN   

    Bootstrap_IC_low  Bootstrap_IC_high  
B0               NaN                NaN  
B1               NaN                NaN  
B2               NaN                NaN  


# AGGREGATING



In [7]:
#train test split
X = df.drop(['mpg', 'model'], axis=1)
y = df['mpg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [ ]:
#train
columnas = [col for col in df.columns if col not in ['mpg', 'model']]

modelos = []
vars_por_iteracion = []

for _ in range(1000):
    vars_i = np.random.choice(columnas, size=3, replace=False)
    X_train_i = X_train[vars_i]

    modelo_i = LinearRegression().fit(X_train_i, y_train)

    modelos.append(modelo_i)
    vars_por_iteracion.append(vars_i)

In [ ]:
#test
predicciones = []

for i in range(1000):
    X_test_i = X_test[vars_por_iteracion[i]]
    y_pred_i = modelos[i].predict(X_test_i)

    predicciones.append(y_pred_i)

# Resultados
predicciones_matrix = np.array(predicciones)
y_hat_vector = predicciones_matrix.mean(axis=0)
r2_ensemble = r2_score(y_test, y_hat_vector)

print(f"R² ensemble (vector agregado):    {r2_ensemble:.4f}")

R² ensemble (vector agregado):    0.7857


# Random Forest

In [41]:
model = RandomForestRegressor(random_state=42)
model.fit(X, y)
y_pred = model.predict(X)
r2_score(y, y_pred)

0.9770244219894207

## Hacer k-folds con k = 10 y usar R2 de pérdida

In [42]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_r2 = cross_val_score(model, X, y, cv=cv, scoring='r2')
cv_scores_mse = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error')

print(f'Mean R2: {cv_scores_r2.mean():.4f}')
print(f'Mean MSE: {-cv_scores_mse.mean():.4f}')

Mean R2: 0.4571
Mean MSE: 5.9571


### Hyperparameter Tuning with Bayesian Search
Since the dataset is small, tuning the forest parameters is crucial to prevent overfitting and improve the R2.

In [43]:
from skopt import BayesSearchCV

# Define search space
search_space = {
    'n_estimators': (3, 100),
    'max_depth': (2, 10),
    'min_samples_split': (2, 10),
    'min_samples_leaf': (1, 5),
    'max_features': ['sqrt', 'log2', None]
}

opt = BayesSearchCV(
    RandomForestRegressor(random_state=42),
    search_space,
    n_iter=32,
    cv=10,
    n_jobs=-1,
    scoring='r2',
    random_state=42
)

opt.fit(X, y)

print(f"Best R2 found: {opt.best_score_:.4f}")
print(f"Best Params: {opt.best_params_}")

Best R2 found: 0.5827
Best Params: OrderedDict({'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 7, 'n_estimators': 59})


In [44]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(model,
                  param_grid = {'max_depth': range(1, 11),
                                'min_samples_split': range(10, 60, 10),
                                'n_estimators': range(1,100,10)},
                  cv=10,
                  scoring='r2')
gs.fit(X, y)


GridSearchCV(cv=10, estimator=RandomForestRegressor(random_state=42),
             param_grid={'max_depth': range(1, 11),
                         'min_samples_split': range(10, 60, 10),
                         'n_estimators': range(1, 100, 10)},
             scoring='r2')

In [62]:
#crear modelo usando parámetros óptimos
new_model = RandomForestRegressor(n_estimators=51,
                               criterion='squared_error',
                               max_depth=4,
                               min_samples_split=10,
                               min_samples_leaf=1,
                               bootstrap=True,
                               oob_score=False,
                               random_state=42,
                               verbose=0)
#Entrenamiento
new_model.fit(X, y)

RandomForestRegressor(max_depth=4, min_samples_split=10, n_estimators=51,
                      random_state=42)

In [64]:
yhat = new_model.predict(X)
R2_score = r2_score(y,yhat)
print('R2:', R2_score)

R2: 0.9077744312015148


## Boosting


In [45]:
from sklearn.ensemble import GradientBoostingRegressor

In [50]:
model_boost = GradientBoostingRegressor(random_state=42)
model_boost.fit(X, y)
y_pred = model_boost.predict(X)
r2_score(y, y_pred)

0.9999297755665103

In [52]:
model_boost.get_params()

{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 42,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [53]:
cv_scores_r2 = cross_val_score(model_boost, X, y, cv=cv, scoring='r2')
cv_scores_mse = cross_val_score(model_boost, X, y, cv=cv, scoring='neg_mean_squared_error')

print(f'Mean R2: {cv_scores_r2.mean():.4f}')
print(f'Mean MSE: {-cv_scores_mse.mean():.4f}')

Mean R2: 0.5640
Mean MSE: 6.4218


In [56]:
from sklearn.model_selection import GridSearchCV

# Using LeaveOneOut for the most robust estimate on 32 samples

param_grid = {
    'n_estimators': [20, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3],
    'subsample': [0.6, 0.8, 1.0],
    'max_features': ['sqrt', None]
}

gb_grid = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

gb_grid.fit(X, y)

GridSearchCV(cv=5, estimator=GradientBoostingRegressor(random_state=42),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [2, 3], 'max_features': ['sqrt', None],
                         'n_estimators': [20, 50, 100],
                         'subsample': [0.6, 0.8, 1.0]},
             scoring='r2')

In [60]:
#crear modelo usando parámetros óptimos
new_model_boost = GradientBoostingRegressor(
                               criterion='squared_error',
                               max_depth=2,
                               max_features='sqrt',
                               min_samples_leaf=1,
                               random_state=42,
                               subsample=0.8,
                               verbose=0)
#Entrenamiento
new_model_boost.fit(X, y)

GradientBoostingRegressor(criterion='squared_error', max_depth=2,
                          max_features='sqrt', random_state=42, subsample=0.8)

In [61]:
yhat = new_model_boost.predict(X)
R2_score = r2_score(y,yhat)
print('R2:', R2_score)

R2: 0.9954913544035305
